# SDWPF single-turbine gap filling

Attach a Kaggle dataset containing both SDWPF CSV files, enable GPU if available, then run all cells. Each episode hides one randomly selected turbine and predicts only that turbine's current Patv and Wspd. The target turbine's full ten-channel record is hidden through its outage. Other turbines can have additional random or block outages; their same-time observed readings are available to the models. This is gap filling, not forecasting. Comparators include persistence, a nearest-peer spatial-persistence baseline, a distance-weighted neighbor mean, project-wide global ridge comparator, Baidu's KDD Cup 2022 BaselineGruModel architecture, an MLP, and matched undirected and wind-directed GCNs. The paper GRU uses its original 144-step, 10-feature encoder, 2-layer 48-unit GRU, 0.05 dropout, and Patv-only projection. Its zero-input decoder is shortened to the selected gap so only the endpoint is scored; it is therefore the actual paper architecture adapted to this imputation protocol, not the paper's original 288-step forecasting experiment. It uses the paper's Adam, 1e-4 starting rate, halving schedule, batch size 32, 10-epoch cap, and patience 3. Its results are reported for Patv only. Source: https://github.com/PaddlePaddle/PaddleSpatial/tree/main/apps/wpf_baseline_gru. The directed graph uses the prior forecast experiment's estimated coordinate rotation/reflection; treat that mapping as a hypothesis. Labels follow the companion notebook's target-validity filters. Use `pilot` for a quick check, `extended` for 30% and 70% random/block peer outages across all three gap lengths, or `full` for three seeds and 0/30/50/70% outage conditions. Results are saved in `/kaggle/working/outputs/`.

In [1]:
import os
SDWPF_GAPFILL_RUN_MODE = 'full'  # pilot, extended, full
os.environ['SDWPF_GAPFILL_RUN_MODE'] = SDWPF_GAPFILL_RUN_MODE


# SDWPF single-turbine gap filling

This experiment reconstructs **one selected turbine's missing current value** at a time.
Other turbines provide same-timestamp readings when available; simulated outages can hide
additional peers. This is imputation, not future forecasting.

For gap length `s`, the selected turbine's readings from `t-s+1` through `t` are hidden.
Its latest eligible self-history is at `t-s`; peer turbines may contribute their values
at `t`. Only the selected turbine's valid `Patv` and `Wspd` labels at `t` are scored.

The task is evaluated at 10-minute, 1-hour, and 6-hour gaps. The split is chronological.
Scaling is fitted on training timestamps only. The graph models and MLP receive exactly
the same current peer information and target-turbine history. The target's entire ten-
channel record is hidden; peer outage scenarios can hide additional complete records.
Compare pure persistence, the old nearest-peer spatial persistence, an eight-neighbor
distance-weighted mean, linear regression, the Baidu KDD Cup 2022
GRU architecture, an MLP, and matched undirected/directed GCNs. The paper GRU's original
Patv-only architecture is retained; its forecast decoder is adapted to endpoint gaps.
Scores exclude invalid target records using the companion notebook's filters.

In [2]:
from __future__ import annotations

import argparse
import json
import os
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

FEATURES = [
    "Patv", "Wspd", "sin_abs_wdir", "cos_abs_wdir", "Etmp", "Itmp",
    "Pab1", "Pab2", "Pab3", "Prtv",
]
TARGETS = ["Patv", "Wspd"]
TARGET_FEATURE_INDEX = np.array([FEATURES.index(name) for name in TARGETS])
GAP_LENGTHS = {"10min": 1, "1hour": 6, "6hour": 36}
PEER_OUTAGE_FRACTIONS = [0.0, 0.30, 0.50, 0.70]
PEER_OUTAGE_PATTERNS = ["random", "block"]
SPLIT_RATIOS = (0.70, 0.10, 0.20)
HISTORY_LENGTH = 12
PAPER_HISTORY_LENGTH = 144
PAPER_GRU_HIDDEN_SIZE = 48
PAPER_GRU_LAYERS = 2
PAPER_GRU_DROPOUT = 0.05
HIDDEN_DIM = 64
BATCH_SIZE = 256
MAX_EPOCHS = 60
PATIENCE = 10
WIND_COORD_ROTATION_DEGREES = 60.0
WIND_COORD_REFLECT_X = True
TRAIN_EPISODES = 20_000
VALIDATION_EPISODES = 10_000
TEST_EPISODES = 20_000
SEEDS = [0, 1, 2]
RUN_MODE = os.environ.get("SDWPF_GAPFILL_RUN_MODE", "extended").lower()
if RUN_MODE not in {"pilot", "extended", "full"}:
    raise ValueError(f"SDWPF_GAPFILL_RUN_MODE must be pilot, extended, or full; got {RUN_MODE!r}.")


def locate_data() -> tuple[Path, Path, Path]:
    input_root = Path("/kaggle/input")
    if input_root.is_dir():
        data_files = list(input_root.rglob("sdwpf_245days_v1.csv"))
        location_files = list(input_root.rglob("sdwpf_baidukddcup2022_turb_location.csv"))
        if not data_files or not location_files:
            raise FileNotFoundError(
                "Attach a Kaggle dataset containing both sdwpf_245days_v1.csv and "
                "sdwpf_baidukddcup2022_turb_location.csv."
            )
        return data_files[0], location_files[0], Path("/kaggle/working/outputs")
    root = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
    data_path = Path(os.environ.get("SDWPF_DATA_PATH", root / "data" / "sdwpf_245days_v1.csv"))
    location_path = Path(
        os.environ.get(
            "SDWPF_LOC_PATH", root / "data" / "sdwpf_baidukddcup2022_turb_location.csv"
        )
    )
    output_path = Path(os.environ.get("SDWPF_OUTPUT_DIR", root / "outputs"))
    return data_path, location_path, output_path


DATA_PATH, LOCATION_PATH, OUTPUT_DIR = locate_data()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def _valid_direction(wdir: np.ndarray, ndir: np.ndarray) -> np.ndarray:
    return (
        np.isfinite(wdir)
        & (wdir >= -180.0)
        & (wdir <= 180.0)
        & np.isfinite(ndir)
        & (ndir >= -720.0)
        & (ndir <= 720.0)
    )


def load_data() -> tuple[np.ndarray, np.ndarray, pd.DataFrame, pd.DataFrame]:
    """Load SDWPF into [time, turbine, feature] values and a native-observation mask."""
    if not DATA_PATH.is_file() or not LOCATION_PATH.is_file():
        raise FileNotFoundError(f"Could not find SDWPF inputs: {DATA_PATH}, {LOCATION_PATH}")
    locations = pd.read_csv(LOCATION_PATH).sort_values("TurbID").reset_index(drop=True)
    turbine_ids = locations["TurbID"].to_numpy()
    if len(np.unique(turbine_ids)) != len(turbine_ids):
        raise ValueError("The location file contains duplicate turbine IDs.")
    usecols = [
        "TurbID", "Day", "Tmstamp", "Wspd", "Wdir", "Ndir", "Etmp", "Itmp",
        "Pab1", "Pab2", "Pab3", "Prtv", "Patv",
    ]
    day_min: int | None = None
    day_max: int | None = None
    for frame in pd.read_csv(DATA_PATH, usecols=["Day"], chunksize=250_000):
        frame_min, frame_max = int(frame["Day"].min()), int(frame["Day"].max())
        day_min = frame_min if day_min is None else min(day_min, frame_min)
        day_max = frame_max if day_max is None else max(day_max, frame_max)
    if day_min is None or day_max is None:
        raise ValueError("The SCADA file contains no records.")
    clock = [f"{hour:02d}:{minute:02d}" for hour in range(24) for minute in range(0, 60, 10)]
    clock_index = {value: index for index, value in enumerate(clock)}
    turbine_index = {int(value): index for index, value in enumerate(turbine_ids)}
    n_times = (day_max - day_min + 1) * len(clock)
    values = np.full((n_times, len(turbine_ids), len(FEATURES)), np.nan, dtype=np.float32)
    seen = np.zeros((n_times, len(turbine_ids)), dtype=bool)
    valid_rows = np.zeros((n_times, len(turbine_ids)), dtype=bool)
    for frame in pd.read_csv(DATA_PATH, usecols=usecols, chunksize=250_000):
        nodes = frame["TurbID"].map(turbine_index)
        times = frame["Tmstamp"].map(clock_index)
        if nodes.isna().any() or times.isna().any():
            raise ValueError("SCADA rows contain an unknown turbine ID or timestamp.")
        node_idx = nodes.to_numpy(dtype=np.int64)
        time_idx = (
            (frame["Day"].to_numpy(dtype=np.int64) - day_min) * len(clock)
            + times.to_numpy(dtype=np.int64)
        )
        if seen[time_idx, node_idx].any():
            raise ValueError("Duplicate turbine/timestamp records were found.")
        seen[time_idx, node_idx] = True
        columns = {
            name: pd.to_numeric(frame[name], errors="coerce").to_numpy(dtype=np.float32)
            for name in [
                "Patv", "Wspd", "Wdir", "Ndir", "Etmp", "Itmp",
                "Pab1", "Pab2", "Pab3", "Prtv",
            ]
        }
        direction_ok = _valid_direction(columns["Wdir"], columns["Ndir"])
        row_complete = np.logical_and.reduce(
            [np.isfinite(column) for column in columns.values()]
        )
        unknown = (
            ((columns["Patv"] <= 0.0) & (columns["Wspd"] > 2.5))
            | (columns["Pab1"] > 89.0)
            | (columns["Pab2"] > 89.0)
            | (columns["Pab3"] > 89.0)
        )
        target_valid = row_complete & direction_ok & ~unknown
        radians = np.deg2rad(
            np.where(direction_ok, columns["Wdir"] + columns["Ndir"], 0.0)
        )
        features = np.stack(
            [
                np.where(direction_ok, np.maximum(columns["Patv"], 0.0), np.nan),
                columns["Wspd"],
                np.where(direction_ok, np.sin(radians), np.nan),
                np.where(direction_ok, np.cos(radians), np.nan),
                columns["Etmp"],
                columns["Itmp"],
                columns["Pab1"],
                columns["Pab2"],
                columns["Pab3"],
                columns["Prtv"],
            ],
            axis=-1,
        )
        values[time_idx, node_idx] = features
        valid_rows[time_idx, node_idx] = target_valid
    if not seen.all():
        raise ValueError(f"The SCADA table is missing {int((~seen).sum())} turbine/time rows.")
    times = pd.DataFrame(
        {
            "Day": np.repeat(np.arange(day_min, day_max + 1), len(clock)),
            "Tmstamp": clock * (day_max - day_min + 1),
        }
    )
    observed = np.isfinite(values)
    # Match the companion gap-filling notebook's row-level validity definition: missing
    # source channels, abnormal direction, stopped turbines, and feathered blades are
    # excluded from both labels and peer inputs. Negative active-power readings are clipped
    # to zero and remain valid unless the stopped-turbine rule applies.
    observed &= valid_rows[:, :, None]
    return values, observed, times, locations


def chronological_boundaries(n_times: int) -> tuple[int, int]:
    steps_per_day = 24 * 6
    if n_times % steps_per_day:
        raise ValueError("The SDWPF timeline must contain complete 10-minute days.")
    days = n_times // steps_per_day
    train_days = int(np.floor(days * SPLIT_RATIOS[0] + 1e-9))
    validation_days = int(np.floor(days * sum(SPLIT_RATIOS[:2]) + 1e-9))
    return train_days * steps_per_day, validation_days * steps_per_day


def fit_scaler(
    values: np.ndarray, observed: np.ndarray, train_end: int
) -> tuple[np.ndarray, np.ndarray]:
    means = np.zeros(values.shape[-1], dtype=np.float32)
    scales = np.ones_like(means)
    for feature in range(values.shape[-1]):
        samples = values[:train_end, :, feature][observed[:train_end, :, feature]]
        if samples.size == 0:
            raise ValueError(f"No training observations for {FEATURES[feature]}.")
        means[feature] = samples.mean(dtype=np.float64)
        scale = samples.std(dtype=np.float64)
        scales[feature] = scale if scale > 1e-6 else 1.0
    return means, scales


def make_outage_mask(
    n_times: int, n_nodes: int, train_end: int, validation_end: int, pattern: str, fraction: float, seed: int
) -> np.ndarray:
    """Mask whole peer records; block mode uses contiguous 2-6 hour outages."""
    result = np.zeros((n_times, n_nodes), dtype=bool)
    if fraction == 0:
        return result
    bounds = [(0, train_end), (train_end, validation_end), (validation_end, n_times)]
    for split_id, (start, stop) in enumerate(bounds):
        rng = np.random.default_rng(seed + 1009 * (split_id + 1))
        local_steps = stop - start
        count = int(round(local_steps * n_nodes * fraction))
        local = np.zeros((local_steps, n_nodes), dtype=bool)
        if pattern == "random":
            local.reshape(-1)[rng.choice(local.size, size=count, replace=False)] = True
        elif pattern == "block":
            # Place contiguous 12-36 step outages and distribute healthy gaps around them.
            budgets = np.full(n_nodes, count // n_nodes, dtype=np.int64)
            if count % n_nodes:
                budgets[rng.permutation(n_nodes)[: count % n_nodes]] += 1
            for node, budget in enumerate(budgets):
                if budget == 0:
                    continue
                block_count = max(1, int(round(int(budget) / 24.0)))
                weights = rng.uniform(0.75, 1.25, size=block_count)
                lengths = np.floor(weights / weights.sum() * int(budget)).astype(np.int64)
                lengths[lengths < 1] = 1
                while int(lengths.sum()) < budget:
                    lengths[int(rng.integers(0, block_count))] += 1
                while int(lengths.sum()) > budget:
                    candidates = np.flatnonzero(lengths > 1)
                    lengths[int(rng.choice(candidates))] -= 1
                gaps = rng.multinomial(
                    local_steps - int(budget),
                    np.full(block_count + 1, 1.0 / (block_count + 1)),
                )
                cursor = int(gaps[0])
                for block_idx, length in enumerate(lengths):
                    block_stop = cursor + int(length)
                    local[cursor:block_stop, node] = True
                    cursor = block_stop + int(gaps[block_idx + 1])
            if int(local.sum()) != count:
                raise AssertionError("Block outage mask did not match its exact missing-cell budget.")
        else:
            raise ValueError(f"Unsupported peer outage pattern {pattern!r}.")
        result[start:stop] = local
    return result


def candidate_pairs(
    observed: np.ndarray, split_start: int, split_stop: int, gap: int
) -> np.ndarray:
    """Return (timestamp, target-node) pairs with valid current and last-pre-gap labels."""
    target_valid = observed[:, :, TARGET_FEATURE_INDEX].all(axis=-1)
    timestamps = np.arange(
        max(split_start + gap, PAPER_HISTORY_LENGTH + gap), split_stop
    )
    valid = target_valid[timestamps] & target_valid[timestamps - gap]
    pairs = np.argwhere(valid)
    if not len(pairs):
        raise ValueError(f"No valid single-turbine examples for gap={gap} in this split.")
    pairs[:, 0] = timestamps[pairs[:, 0]]
    pairs = pairs.astype(np.int64)
    return np.column_stack([pairs, np.full(len(pairs), gap, dtype=np.int64)])


def sample_pairs(
    candidates: np.ndarray, count: int, seed: int
) -> np.ndarray:
    rng = np.random.default_rng(seed)
    if count >= len(candidates):
        return candidates.copy()
    return candidates[rng.choice(len(candidates), size=count, replace=False)]


def build_knn_adjacency(locations: pd.DataFrame, k: int = 8) -> tuple[np.ndarray, np.ndarray]:
    coords = locations[["x", "y"]].to_numpy(dtype=np.float32)
    distances = np.linalg.norm(coords[:, None, :] - coords[None, :, :], axis=-1)
    np.fill_diagonal(distances, np.inf)
    directed = np.zeros_like(distances, dtype=np.float32)
    for node in range(len(coords)):
        directed[node, np.argpartition(distances[node], k)[:k]] = 1.0
    adjacency = np.maximum(directed, directed.T)
    sigma = max(float(np.median(distances[np.isfinite(distances)])), 1.0)
    weights = adjacency * np.exp(-distances / sigma)
    np.fill_diagonal(weights, 0.0)
    weights += np.eye(len(coords), dtype=np.float32)
    weights /= np.maximum(weights.sum(axis=1, keepdims=True), 1e-8)
    return weights.astype(np.float32), distances.astype(np.float32)


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_batch(
    values: np.ndarray,
    observed: np.ndarray,
    outage_mask: np.ndarray,
    pairs: np.ndarray,
    device: torch.device,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    """Build current farm snapshot, target pre-gap history, target IDs and labels."""
    times, targets, gaps = pairs[:, 0], pairs[:, 1], pairs[:, 2]
    batch_size, nodes = len(pairs), values.shape[1]
    current_values = values[times].copy()
    current_mask = observed[times].copy() & ~outage_mask[times, :, None]
    current_mask[np.arange(batch_size), targets] = False
    current_values[~current_mask] = 0.0
    node_features = np.concatenate(
        [
            current_values,
            current_mask.astype(np.float32),
            np.zeros((batch_size, nodes, 1), dtype=np.float32),
        ],
        axis=-1,
    )
    node_features[np.arange(batch_size), targets, -1] = 1.0

    history_indices = times[:, None] - gaps[:, None] - np.arange(HISTORY_LENGTH)[None, :]
    history_values = values[
        history_indices[:, :, None], targets[:, None, None], TARGET_FEATURE_INDEX[None, None, :]
    ]
    history_mask = observed[
        history_indices[:, :, None], targets[:, None, None], TARGET_FEATURE_INDEX[None, None, :]
    ] & ~outage_mask[history_indices, targets[:, None]][:, :, None]
    history = np.concatenate(
        [
            np.where(history_mask, history_values, 0.0),
            history_mask.astype(np.float32),
        ],
        axis=-1,
    ).reshape(batch_size, -1)
    labels = values[times[:, None], targets[:, None], TARGET_FEATURE_INDEX[None, :]]
    return (
        torch.from_numpy(node_features).to(device),
        torch.from_numpy(history).to(device),
        torch.from_numpy(targets).to(device),
        torch.from_numpy(labels.astype(np.float32)).to(device),
    )


def make_paper_history(
    values: np.ndarray,
    observed: np.ndarray,
    outage_mask: np.ndarray,
    pairs: np.ndarray,
    device: torch.device,
) -> torch.Tensor:
    """Return 144 masked own-turbine steps for the adapted 2022 paper GRU."""
    times, targets, gaps = pairs[:, 0], pairs[:, 1], pairs[:, 2]
    indices = (
        times[:, None]
        - gaps[:, None]
        - (PAPER_HISTORY_LENGTH - 1)
        + np.arange(PAPER_HISTORY_LENGTH, dtype=np.int64)[None, :]
    )
    target_values = values[
        indices[:, :, None], targets[:, None, None], np.arange(len(FEATURES))[None, None, :]
    ]
    target_mask = observed[
        indices[:, :, None], targets[:, None, None], np.arange(len(FEATURES))[None, None, :]
    ] & ~outage_mask[indices, targets[:, None]][:, :, None]
    features = np.where(target_mask, target_values, 0.0)
    return torch.from_numpy(features.astype(np.float32)).to(device)


class GapFillMLP(nn.Module):
    """Non-graph comparator receiving the same current farm snapshot and target history."""

    def __init__(self, node_count: int, node_features: int, history_features: int):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(node_count * node_features + history_features, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, len(TARGETS)),
        )

    def forward(
        self,
        node_x: torch.Tensor,
        history: torch.Tensor,
        target_ids: torch.Tensor,
        adjacency: torch.Tensor | None = None,
    ) -> torch.Tensor:
        del target_ids, adjacency
        return self.network(torch.cat([node_x.flatten(1), history], dim=1))


class GapFillGCN(nn.Module):
    """Two-hop spatial encoder with target-specific history and output."""

    def __init__(self, node_features: int, history_features: int, hidden_dim: int):
        super().__init__()
        self.input = nn.Linear(node_features, hidden_dim)
        self.spatial1 = nn.Linear(hidden_dim, hidden_dim)
        self.spatial2 = nn.Linear(hidden_dim, hidden_dim)
        self.head = nn.Sequential(
            nn.Linear(hidden_dim + history_features, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, len(TARGETS)),
        )

    def forward(
        self,
        node_x: torch.Tensor,
        history: torch.Tensor,
        target_ids: torch.Tensor,
        adjacency: torch.Tensor,
    ) -> torch.Tensor:
        hidden = F.relu(self.input(node_x))
        hidden = F.relu(self.spatial1(torch.matmul(adjacency, hidden)))
        hidden = F.relu(self.spatial2(torch.matmul(adjacency, hidden)))
        target_embedding = hidden[torch.arange(len(target_ids), device=target_ids.device), target_ids]
        return self.head(torch.cat([target_embedding, history], dim=-1))


class GapFillDirectedGCN(GapFillGCN):
    """Same GCN as the undirected comparator; only its episode graph is directed."""


class PaperGRU2022(nn.Module):
    """PyTorch equivalent of Baidu's KDD Cup 2022 BaselineGruModel architecture."""

    def __init__(self, input_features: int = len(FEATURES)):
        super().__init__()
        self.gru = nn.GRU(
            input_size=input_features,
            hidden_size=PAPER_GRU_HIDDEN_SIZE,
            num_layers=PAPER_GRU_LAYERS,
            batch_first=True,
        )
        self.dropout = nn.Dropout(PAPER_GRU_DROPOUT)
        self.projection = nn.Linear(PAPER_GRU_HIDDEN_SIZE, 1)

    def forward(self, history: torch.Tensor, output_len: int) -> torch.Tensor:
        decoder_zeros = history.new_zeros(
            (history.shape[0], output_len, history.shape[2])
        )
        sequence = torch.cat([history, decoder_zeros], dim=1)
        decoded, _ = self.gru(sequence)
        projected = self.projection(self.dropout(decoded))
        return projected[:, -1, :]


def build_directed_adjacency(
    node_x: torch.Tensor,
    coords: torch.Tensor,
    undirected_adjacency: torch.Tensor,
    feature_means: torch.Tensor,
    feature_scales: torch.Tensor,
) -> torch.Tensor:
    """Weight nearby source nodes by current observed wind-flow alignment."""
    feature_count = len(FEATURES)
    wind_mask = (node_x[:, :, feature_count + 2] > 0.5) & (
        node_x[:, :, feature_count + 3] > 0.5
    )
    sine = node_x[:, :, 2] * feature_scales[2] + feature_means[2]
    cosine = node_x[:, :, 3] * feature_scales[3] + feature_means[3]
    count = wind_mask.sum(dim=1, keepdim=True).clamp_min(1)
    mean_sine = torch.where(wind_mask, sine, 0.0).sum(dim=1, keepdim=True) / count
    mean_cosine = torch.where(wind_mask, cosine, 0.0).sum(dim=1, keepdim=True) / count
    flow = torch.cat([-mean_sine, -mean_cosine], dim=-1)

    angle = np.deg2rad(WIND_COORD_ROTATION_DEGREES)
    transform = torch.tensor(
        [
            [np.cos(angle), -np.sin(angle)],
            [np.sin(angle), np.cos(angle)],
        ],
        dtype=coords.dtype,
        device=coords.device,
    )
    if WIND_COORD_REFLECT_X:
        transform = transform @ torch.diag(
            torch.tensor([-1.0, 1.0], dtype=coords.dtype, device=coords.device)
        )
    compass_coords = coords @ transform.T
    delta = compass_coords[:, None, :] - compass_coords[None, :, :]
    distance = torch.linalg.vector_norm(delta, dim=-1).clamp_min(1e-8)
    bearing = delta / distance[:, :, None]
    flow_norm = torch.linalg.vector_norm(flow, dim=-1, keepdim=True)
    flow = torch.where(flow_norm > 1e-6, flow / flow_norm.clamp_min(1e-8), 0.0)
    alignment = torch.relu(
        torch.einsum("bd,ijd->bij", flow, bearing)
    )

    base_weights = 0.5 * (undirected_adjacency + undirected_adjacency.T)
    base_weights = base_weights.clone()
    base_weights.fill_diagonal_(0.0)
    directed = alignment * base_weights[None, :, :]
    directed = directed + torch.eye(
        coords.shape[0], dtype=node_x.dtype, device=node_x.device
    )[None, :, :]
    return directed / directed.sum(dim=-1, keepdim=True).clamp_min(1e-8)


def fit_ridge(
    values: np.ndarray,
    observed: np.ndarray,
    outage_mask: np.ndarray,
    pairs: np.ndarray,
    device: torch.device,
) -> np.ndarray:
    """Fit a closed-form linear-both reference on the same episode inputs."""
    feature_count = values.shape[-1]
    node_count = values.shape[1]
    history_features = HISTORY_LENGTH * len(TARGETS) * 2
    id_features = node_count
    design_count = node_count * (feature_count * 2 + 1) + history_features + id_features + 1
    xtx = np.zeros((design_count, design_count), dtype=np.float64)
    xty = np.zeros((design_count, len(TARGETS)), dtype=np.float64)
    batch_size = 512
    for start in range(0, len(pairs), batch_size):
        batch = pairs[start : start + batch_size]
        node_x, history, target_ids, labels = make_batch(
            values, observed, outage_mask, batch, device
        )
        identity = F.one_hot(target_ids, num_classes=node_count).float()
        design = torch.cat(
            [node_x.flatten(1), history, identity, torch.ones((len(batch), 1), device=device)],
            dim=1,
        ).cpu().numpy().astype(np.float64)
        target = labels.cpu().numpy().astype(np.float64)
        xtx += design.T @ design
        xty += design.T @ target
    regularizer = np.eye(design_count, dtype=np.float64) * 1e-2
    regularizer[-1, -1] = 0.0
    return np.linalg.solve(xtx + regularizer, xty)


def ridge_design(
    node_x: torch.Tensor,
    history: torch.Tensor,
    target_ids: torch.Tensor,
) -> torch.Tensor:
    ids = F.one_hot(target_ids, num_classes=node_x.shape[1]).float()
    return torch.cat(
        [node_x.flatten(1), history, ids, torch.ones((len(target_ids), 1), device=node_x.device)],
        dim=1,
    )


@dataclass(frozen=True)
class RunSettings:
    gaps: tuple[int, ...]
    conditions: tuple[tuple[str, float], ...]
    seeds: tuple[int, ...]
    train_episodes: int
    validation_episodes: int
    test_episodes: int
    epochs: int


def run_settings() -> RunSettings:
    if RUN_MODE == "pilot":
        return RunSettings((6,), (("random", 0.30),), (0,), 1024, 512, 512, 3)
    if RUN_MODE == "extended":
        return RunSettings(
            (1, 6, 36),
            (
                ("random", 0.30),
                ("block", 0.30),
                ("random", 0.70),
                ("block", 0.70),
            ),
            (0,),
            10_000,
            5_000,
            5_000,
            15,
        )
    conditions = (("random", 0.0),) + tuple(
        (pattern, fraction)
        for fraction in PEER_OUTAGE_FRACTIONS[1:]
        for pattern in PEER_OUTAGE_PATTERNS
    )
    return RunSettings(
        (1, 6, 36), conditions, tuple(SEEDS), TRAIN_EPISODES,
        VALIDATION_EPISODES, TEST_EPISODES, MAX_EPOCHS
    )


def evaluate_baselines(
    values: np.ndarray,
    observed: np.ndarray,
    outage_mask: np.ndarray,
    pairs: np.ndarray,
    adjacency: np.ndarray,
    means: np.ndarray,
    scales: np.ndarray,
    ridge_weights: np.ndarray,
) -> dict[str, np.ndarray]:
    node_x, history, target_ids, _ = make_batch(
        values, observed, outage_mask, pairs, DEVICE
    )
    times = pairs[:, 0]
    targets = pairs[:, 1]
    gaps = pairs[:, 2]
    persistence_indices = (
        times[:, None]
        - gaps[:, None]
        - np.arange(PAPER_HISTORY_LENGTH, dtype=np.int64)[None, :]
    )
    persistence_values = values[
        persistence_indices[:, :, None],
        targets[:, None, None],
        TARGET_FEATURE_INDEX[None, None, :],
    ]
    persistence_mask = observed[
        persistence_indices[:, :, None],
        targets[:, None, None],
        TARGET_FEATURE_INDEX[None, None, :],
    ] & ~outage_mask[persistence_indices, targets[:, None]][:, :, None]
    latest_positions = np.where(
        persistence_mask,
        np.arange(PAPER_HISTORY_LENGTH)[None, :, None],
        PAPER_HISTORY_LENGTH,
    ).min(axis=1)
    latest_indices = np.minimum(latest_positions, PAPER_HISTORY_LENGTH - 1)[:, None, :]
    latest = np.take_along_axis(persistence_values, latest_indices, axis=1)[:, 0, :]
    latest = np.where(latest_positions < PAPER_HISTORY_LENGTH, latest, 0.0)
    outputs = {"persistence": latest.copy()}
    raw_features = node_x[:, :, : len(FEATURES)]
    raw_masks = node_x[:, :, len(FEATURES) : 2 * len(FEATURES)]
    neighbor_adjacency = adjacency.copy()
    np.fill_diagonal(neighbor_adjacency, 0.0)
    binary_neighbors = torch.as_tensor(
        neighbor_adjacency > 0.0, device=DEVICE
    )[target_ids]
    spatial_predictions = torch.zeros(
        (len(pairs), len(TARGETS)), dtype=torch.float32, device=DEVICE
    )
    for target_idx, feature_idx in enumerate(TARGET_FEATURE_INDEX):
        available_neighbors = binary_neighbors * raw_masks[:, :, feature_idx]
        neighbor_count = available_neighbors.sum(dim=1)
        neighbor_sum = (available_neighbors * raw_features[:, :, feature_idx]).sum(dim=1)
        neighbor_mean = neighbor_sum / neighbor_count.clamp_min(1.0)

        available_farm = raw_masks[:, :, feature_idx]
        farm_count = available_farm.sum(dim=1)
        farm_sum = (available_farm * raw_features[:, :, feature_idx]).sum(dim=1)
        farm_mean = farm_sum / farm_count.clamp_min(1.0)
        spatial_predictions[:, target_idx] = torch.where(
            neighbor_count > 0, neighbor_mean, farm_mean
        )
    outputs["spatial_persistence_k8"] = spatial_predictions.cpu().numpy()

    weighted = torch.as_tensor(adjacency, device=DEVICE)[target_ids]
    for target_idx, feature_idx in enumerate(TARGET_FEATURE_INDEX):
        neighbor_weights = weighted * raw_masks[:, :, feature_idx]
        estimate = (neighbor_weights * raw_features[:, :, feature_idx]).sum(1)
        estimate /= neighbor_weights.sum(1).clamp_min(1e-8)
        outputs.setdefault("knn_mean_k8", np.zeros((len(pairs), len(TARGETS)), np.float32))
        outputs["knn_mean_k8"][:, target_idx] = estimate.cpu().numpy()
    design = ridge_design(node_x, history, target_ids).cpu().numpy()
    outputs["linear_both_global_ridge"] = design @ ridge_weights
    return outputs


def train_model(
    model: nn.Module,
    train_pairs: np.ndarray,
    validation_pairs: np.ndarray,
    values: np.ndarray,
    observed: np.ndarray,
    outage_mask: np.ndarray,
    adjacency: torch.Tensor,
    coords: torch.Tensor,
    feature_means: torch.Tensor,
    feature_scales: torch.Tensor,
    settings: RunSettings,
    seed: int,
) -> nn.Module:
    set_seed(seed)
    model = model.to(DEVICE)
    is_paper_gru = isinstance(model, PaperGRU2022)
    batch_size = 32 if is_paper_gru else BATCH_SIZE
    max_epochs = min(settings.epochs, 10) if is_paper_gru else settings.epochs
    patience = 3 if is_paper_gru else PATIENCE
    optimizer = (
        torch.optim.Adam(model.parameters(), lr=1e-4)
        if is_paper_gru
        else torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-4)
    )
    best_loss = float("inf")
    best_state: dict[str, torch.Tensor] | None = None
    wait = 0
    for epoch in range(max_epochs):
        model.train()
        order = np.random.default_rng(seed + epoch).permutation(len(train_pairs))
        for start in range(0, len(order), batch_size):
            batch = train_pairs[order[start : start + batch_size]]
            prediction, labels = run_episode_model(
                model,
                values,
                observed,
                outage_mask,
                batch,
                adjacency,
                coords,
                feature_means,
                feature_scales,
            )
            loss = F.mse_loss(prediction, labels)
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 50.0 if is_paper_gru else 1.0)
            optimizer.step()

        model.eval()
        total_loss = 0.0
        total_count = 0
        with torch.no_grad():
            for start in range(0, len(validation_pairs), batch_size):
                batch = validation_pairs[start : start + batch_size]
                prediction, labels = run_episode_model(
                    model,
                    values,
                    observed,
                    outage_mask,
                    batch,
                    adjacency,
                    coords,
                    feature_means,
                    feature_scales,
                )
                total_loss += float(F.mse_loss(prediction, labels, reduction="sum"))
                total_count += labels.numel()
        validation_loss = total_loss / max(total_count, 1)
        print(f"  epoch {epoch + 1:02d}/{max_epochs}: validation MSE={validation_loss:.5f}")
        if validation_loss < best_loss - 1e-5:
            best_loss = validation_loss
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break
        if is_paper_gru:
            for parameter_group in optimizer.param_groups:
                parameter_group["lr"] *= 0.5
    if best_state is None:
        raise RuntimeError("Training failed to produce a validation checkpoint.")
    model.load_state_dict(best_state)
    return model


def predict_model(
    model: nn.Module,
    values: np.ndarray,
    observed: np.ndarray,
    outage_mask: np.ndarray,
    pairs: np.ndarray,
    adjacency: torch.Tensor,
    coords: torch.Tensor,
    feature_means: torch.Tensor,
    feature_scales: torch.Tensor,
) -> np.ndarray:
    model.eval()
    predictions: list[np.ndarray] = []
    with torch.no_grad():
        for start in range(0, len(pairs), BATCH_SIZE):
            prediction, _ = run_episode_model(
                model,
                values,
                observed,
                outage_mask,
                pairs[start : start + BATCH_SIZE],
                adjacency,
                coords,
                feature_means,
                feature_scales,
            )
            predictions.append(prediction.cpu().numpy())
    return np.concatenate(predictions, axis=0)


def run_episode_model(
    model: nn.Module,
    values: np.ndarray,
    observed: np.ndarray,
    outage_mask: np.ndarray,
    pairs: np.ndarray,
    adjacency: torch.Tensor,
    coords: torch.Tensor,
    feature_means: torch.Tensor,
    feature_scales: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor]:
    node_x, history, target_ids, labels = make_batch(
        values, observed, outage_mask, pairs, DEVICE
    )
    if isinstance(model, PaperGRU2022):
        paper_history = make_paper_history(
            values, observed, outage_mask, pairs, DEVICE
        )
        return model(paper_history, int(pairs[0, 2])), labels[:, :1]
    if isinstance(model, GapFillDirectedGCN):
        adjacency = build_directed_adjacency(
            node_x, coords, adjacency, feature_means, feature_scales
        )
    return model(node_x, history, target_ids, adjacency), labels


def score_predictions(
    predictions: np.ndarray,
    values: np.ndarray,
    pairs: np.ndarray,
    means: np.ndarray,
    scales: np.ndarray,
    target_indices: tuple[int, ...] | None = None,
) -> dict[str, tuple[float, float]]:
    target_indices = target_indices or tuple(range(len(TARGETS)))
    selected_features = TARGET_FEATURE_INDEX[list(target_indices)]
    selected_targets = [TARGETS[index] for index in target_indices]
    truth = values[
        pairs[:, 0, None], pairs[:, 1, None], selected_features[None, :]
    ]
    prediction_real = predictions * scales[selected_features] + means[selected_features]
    truth_real = truth * scales[selected_features] + means[selected_features]
    return {
        target: (
            float(np.mean(np.abs(prediction_real[:, index] - truth_real[:, index]))),
            float(np.sqrt(np.mean((prediction_real[:, index] - truth_real[:, index]) ** 2))),
        )
        for index, target in enumerate(selected_targets)
    }


def run_study() -> pd.DataFrame:
    print(f"Single-turbine gap filling | mode={RUN_MODE} | device={DEVICE}")
    values, observed, _, locations = load_data()
    train_end, validation_end = chronological_boundaries(len(values))
    means, scales = fit_scaler(values, observed, train_end)
    normalized = np.where(observed, (values - means) / scales, 0.0).astype(np.float32)
    # Native observation masks remain separate from standardized fill values.
    node_count = values.shape[1]
    adjacency_array, _ = build_knn_adjacency(locations, k=8)
    adjacency = torch.from_numpy(adjacency_array).to(DEVICE)
    coords = torch.as_tensor(
        locations[["x", "y"]].to_numpy(dtype=np.float32), device=DEVICE
    )
    feature_means = torch.as_tensor(means, device=DEVICE)
    feature_scales = torch.as_tensor(scales, device=DEVICE)
    settings = run_settings()
    rows: list[dict[str, Any]] = []
    for gap in settings.gaps:
        split_candidates = {
            "train": candidate_pairs(observed, 0, train_end, gap),
            "validation": candidate_pairs(observed, train_end, validation_end, gap),
            "test": candidate_pairs(observed, validation_end, len(values), gap),
        }
        for pattern, fraction in settings.conditions:
            for seed in settings.seeds:
                print(f"\nScenario: gap={gap} steps, peer pattern={pattern}, missing={fraction:.0%}, seed={seed}")
                outage_mask = make_outage_mask(
                    len(values), node_count, train_end, validation_end, pattern, fraction, seed
                )
                pair_sets = {
                    "train": sample_pairs(split_candidates["train"], settings.train_episodes, seed + 11),
                    "validation": sample_pairs(
                        split_candidates["validation"], settings.validation_episodes, seed + 23
                    ),
                    "test": sample_pairs(split_candidates["test"], settings.test_episodes, seed + 37),
                }
                # All models receive the same sampled episodes and only test the selected turbine.
                ridge_weights = fit_ridge(
                    normalized,
                    observed,
                    outage_mask,
                    pair_sets["train"],
                    DEVICE,
                )
                baseline_predictions = evaluate_baselines(
                    normalized,
                    observed,
                    outage_mask,
                    pair_sets["test"],
                    adjacency_array,
                    means,
                    scales,
                    ridge_weights,
                )
                for model_name, prediction in baseline_predictions.items():
                    metrics = score_predictions(prediction, normalized, pair_sets["test"], means, scales)
                    for target, (mae, rmse) in metrics.items():
                        rows.append(
                            {
                                "gap_steps": gap,
                                "gap_label": {1: "10min", 6: "1hour", 36: "6hour"}[gap],
                                "peer_outage_pattern": pattern,
                                "peer_outage_fraction": fraction,
                                "seed": seed,
                                "model": model_name,
                                "target": target,
                                "mae": mae,
                                "rmse": rmse,
                                "test_episodes": len(pair_sets["test"]),
                                "scored_turbines_per_episode": 1,
                            }
                        )
                for model_name in (
                    "paper_gru_2022",
                    "mlp",
                    "undirected_gcn",
                    "directed_gcn",
                ):
                    model_offsets = {
                        "paper_gru_2022": 101,
                        "mlp": 202,
                        "undirected_gcn": 303,
                        "directed_gcn": 404,
                    }
                    model_seed = seed + model_offsets[model_name]
                    set_seed(model_seed)
                    if model_name == "paper_gru_2022":
                        model = PaperGRU2022(len(FEATURES))
                    elif model_name == "mlp":
                        model = GapFillMLP(
                            node_count,
                            normalized.shape[-1] * 2 + 1,
                            HISTORY_LENGTH * len(TARGETS) * 2,
                        )
                    elif model_name == "undirected_gcn":
                        model = GapFillGCN(
                            normalized.shape[-1] * 2 + 1,
                            HISTORY_LENGTH * len(TARGETS) * 2,
                            HIDDEN_DIM,
                        )
                    else:
                        model = GapFillDirectedGCN(
                            normalized.shape[-1] * 2 + 1,
                            HISTORY_LENGTH * len(TARGETS) * 2,
                            HIDDEN_DIM,
                        )
                    print(f"Training {model_name}")
                    trained = train_model(
                        model,
                        pair_sets["train"],
                        pair_sets["validation"],
                        normalized,
                        observed,
                        outage_mask,
                        adjacency,
                        coords,
                        feature_means,
                        feature_scales,
                        RunSettings(
                            settings.gaps,
                            settings.conditions,
                            settings.seeds,
                            settings.train_episodes,
                            settings.validation_episodes,
                            settings.test_episodes,
                            settings.epochs,
                        ),
                        model_seed,
                    )
                    prediction = predict_model(
                        trained,
                        normalized,
                        observed,
                        outage_mask,
                        pair_sets["test"],
                        adjacency,
                        coords,
                        feature_means,
                        feature_scales,
                    )
                    model_targets = (0,) if model_name == "paper_gru_2022" else None
                    metrics = score_predictions(
                        prediction,
                        normalized,
                        pair_sets["test"],
                        means,
                        scales,
                        model_targets,
                    )
                    for target, (mae, rmse) in metrics.items():
                        rows.append(
                            {
                                "gap_steps": gap,
                                "gap_label": {1: "10min", 6: "1hour", 36: "6hour"}[gap],
                                "peer_outage_pattern": pattern,
                                "peer_outage_fraction": fraction,
                                "seed": seed,
                                "model": model_name,
                                "target": target,
                                "mae": mae,
                                "rmse": rmse,
                                "test_episodes": len(pair_sets["test"]),
                                "scored_turbines_per_episode": 1,
                            }
                        )
                OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
                results = pd.DataFrame(rows)
                results.to_csv(OUTPUT_DIR / "sdwpf_single_turbine_gapfill_results.csv", index=False)
                print(results.tail(10).to_string(index=False))
    results = pd.DataFrame(rows)
    summary = (
        results.groupby(
            ["gap_steps", "gap_label", "peer_outage_pattern", "peer_outage_fraction", "model", "target"],
            as_index=False,
        )
        .agg(mae_mean=("mae", "mean"), mae_std=("mae", "std"), rmse_mean=("rmse", "mean"), rmse_std=("rmse", "std"))
    )
    summary.to_csv(OUTPUT_DIR / "sdwpf_single_turbine_gapfill_summary.csv", index=False)
    print(f"\nSaved results to {OUTPUT_DIR}")
    print(summary.to_string(index=False))
    return results


def notebook_cells() -> list[tuple[str, str]]:
    source_path = Path(__file__).resolve()
    source = source_path.read_text(encoding="utf-8")
    import re

    source = re.sub(
        r'(?m)^if __name__ == "__main__":\r?\n    main\(\)\r?\n?',
        "",
        source,
    )
    markers = list(re.finditer(r"(?m)^# %%([^\r\n]*)\r?\n", source))
    cells: list[tuple[str, str]] = []
    for index, marker in enumerate(markers):
        end = markers[index + 1].start() if index + 1 < len(markers) else len(source)
        text = source[marker.end() : end].strip("\r\n ")
        if not text:
            continue
        if marker.group(1).strip() == "[markdown]":
            markdown = "\n".join(
                line[2:] if line.startswith("# ") else ("" if line == "#" else line)
                for line in text.splitlines()
            ).strip()
            cells.append(("markdown", markdown))
        else:
            cells.append(("code", text))
    return cells


def create_kaggle_notebook() -> Path:
    import nbformat

    notebook = nbformat.v4.new_notebook()
    notebook.cells = [
        nbformat.v4.new_markdown_cell(
            "# SDWPF single-turbine gap filling\n\n"
            "Attach a Kaggle dataset containing both SDWPF CSV files, enable GPU if available, "
            "then run all cells. Each episode hides one randomly selected turbine and predicts "
            "only that turbine's current Patv and Wspd. The target turbine's full ten-channel "
            "record is hidden through its outage. Other turbines can have additional random or "
            "block outages; their same-time observed readings are available to the models. "
            "This is gap filling, not forecasting. Comparators include persistence, a "
            "nearest-peer spatial-persistence baseline, a distance-weighted neighbor mean, "
            "project-wide global ridge comparator, Baidu's KDD Cup 2022 BaselineGruModel "
            "architecture, an MLP, and matched undirected and wind-directed GCNs. The "
            "paper GRU uses its original 144-step, 10-feature encoder, 2-layer 48-unit "
            "GRU, 0.05 dropout, and Patv-only projection. Its zero-input decoder is shortened "
            "to the selected gap so only the endpoint is scored; it is therefore the actual "
            "paper architecture adapted to this imputation protocol, not the paper's original "
            "288-step forecasting experiment. It uses the paper's Adam, 1e-4 starting rate, "
            "halving schedule, batch size 32, 10-epoch cap, and patience 3. Its results are "
            "reported for Patv only. Source: "
            "https://github.com/PaddlePaddle/PaddleSpatial/tree/main/apps/wpf_baseline_gru. "
            "The directed graph uses the prior forecast experiment's estimated coordinate "
            "rotation/reflection; treat that mapping as a hypothesis. "
            "Labels follow the companion notebook's target-validity filters. Use "
            "`pilot` for a quick check, `extended` for 30% and 70% random/block peer outages "
            "across all three gap lengths, or `full` for three seeds and 0/30/50/70% outage "
            "conditions. "
            "Results are saved in `/kaggle/working/outputs/`."
        ),
        nbformat.v4.new_code_cell(
            "import os\n"
            "SDWPF_GAPFILL_RUN_MODE = 'extended'  # pilot, extended, full\n"
            "os.environ['SDWPF_GAPFILL_RUN_MODE'] = SDWPF_GAPFILL_RUN_MODE\n"
        ),
    ]
    for cell_type, source in notebook_cells():
        if cell_type == "markdown":
            notebook.cells.append(nbformat.v4.new_markdown_cell(source))
        else:
            notebook.cells.append(nbformat.v4.new_code_cell(source))
    notebook.cells.append(nbformat.v4.new_code_cell("results = run_study()"))
    notebook.metadata = {
        "kernelspec": {"display_name": "Python 3", "language": "python", "name": "python3"},
        "language_info": {"name": "python", "version": "3.12"},
    }
    destination = Path(__file__).resolve().parent / "notebooks" / "SDWPF_single_turbine_gapfill_kaggle.ipynb"
    destination.parent.mkdir(parents=True, exist_ok=True)
    nbformat.write(notebook, destination)
    return destination


def run_smoke_tests() -> None:
    values = np.arange(60 * 3 * len(FEATURES), dtype=np.float32).reshape(60, 3, len(FEATURES))
    observed = np.ones_like(values, dtype=bool)
    observed[10, 1, 0] = False
    values[10, 1, 0] = np.nan
    pairs = np.array([[40, 1, 6], [41, 2, 1]], dtype=np.int64)
    outages = np.zeros((60, 3), dtype=bool)
    outages[40, 2] = True
    outages[34, 1] = True
    node_x, history, target_ids, labels = make_batch(
        values, observed, outages, pairs, torch.device("cpu")
    )
    assert node_x.shape == (2, 3, len(FEATURES) * 2 + 1)
    assert history.shape == (2, HISTORY_LENGTH * len(TARGETS) * 2)
    assert target_ids.tolist() == [1, 2]
    assert labels.shape == (2, 2)
    assert torch.all(node_x[0, 1, : len(FEATURES) * 2] == 0)  # target channels are hidden
    assert torch.all(node_x[0, 2, : len(FEATURES) * 2] == 0)  # peer outage channels are hidden
    assert node_x[0, 1, -1] == 1
    assert torch.all(history[0, :4] == 0)  # a pre-gap synthetic outage masks own-history values
    adjacency, _ = build_knn_adjacency(
        pd.DataFrame({"x": [0.0, 1.0, 0.0], "y": [0.0, 0.0, 1.0]}), k=1
    )
    assert adjacency.shape == (3, 3) and np.isfinite(adjacency).all()
    outage_a = make_outage_mask(120, 4, 84, 96, "block", 0.30, 71)
    outage_b = make_outage_mask(120, 4, 84, 96, "block", 0.30, 71)
    assert np.array_equal(outage_a, outage_b)
    for start, stop in ((0, 84), (84, 96), (96, 120)):
        assert int(outage_a[start:stop].sum()) == round(0.30 * (stop - start) * 4)
    for pattern in PEER_OUTAGE_PATTERNS:
        outage_70 = make_outage_mask(120, 4, 84, 96, pattern, 0.70, 72)
        for start, stop in ((0, 84), (84, 96), (96, 120)):
            assert int(outage_70[start:stop].sum()) == round(
                0.70 * (stop - start) * 4
            )
    model = GapFillGCN(len(FEATURES) * 2 + 1, HISTORY_LENGTH * len(TARGETS) * 2, 8)
    output = model(
        node_x,
        history,
        target_ids,
        torch.from_numpy(adjacency.astype(np.float32)),
    )
    assert output.shape == (2, 2) and torch.isfinite(output).all()
    directed = build_directed_adjacency(
        node_x,
        torch.tensor([[0.0, 0.0], [1.0, 0.0], [0.0, 1.0]]),
        torch.from_numpy(adjacency.astype(np.float32)),
        torch.zeros(len(FEATURES)),
        torch.ones(len(FEATURES)),
    )
    assert directed.shape == (len(pairs), *adjacency.shape)
    assert torch.allclose(directed.sum(dim=-1), torch.ones(2, 3), atol=1e-6)

    paper_values = np.ones((220, 3, len(FEATURES)), dtype=np.float32)
    paper_observed = np.ones_like(paper_values, dtype=bool)
    paper_outages = np.zeros((220, 3), dtype=bool)
    paper_pairs = np.array([[180, 1, 6], [181, 2, 1]], dtype=np.int64)
    paper_history = make_paper_history(
        paper_values, paper_observed, paper_outages, paper_pairs, DEVICE
    )
    assert paper_history.shape == (2, PAPER_HISTORY_LENGTH, len(FEATURES))
    assert torch.allclose(
        paper_history[0, 0, 0],
        torch.tensor(paper_values[31, 1, 0], device=DEVICE),
    )
    assert torch.allclose(
        paper_history[0, -1, 0],
        torch.tensor(paper_values[174, 1, 0], device=DEVICE),
    )
    paper_model = PaperGRU2022().to(DEVICE)
    paper_output = paper_model(paper_history, output_len=6)
    assert paper_output.shape == (2, 1) and torch.isfinite(paper_output).all()
    paper_episode_output, _ = run_episode_model(
        paper_model,
        paper_values,
        paper_observed,
        paper_outages,
        paper_pairs,
        torch.from_numpy(adjacency.astype(np.float32)).to(DEVICE),
        torch.tensor([[0.0, 0.0], [1.0, 0.0], [0.0, 1.0]], device=DEVICE),
        torch.zeros(len(FEATURES), device=DEVICE),
        torch.ones(len(FEATURES), device=DEVICE),
    )
    assert paper_episode_output.shape == (2, 1)
    directed_model = GapFillDirectedGCN(
        len(FEATURES) * 2 + 1, HISTORY_LENGTH * len(TARGETS) * 2, 8
    ).to(DEVICE)
    directed_output, _ = run_episode_model(
        directed_model,
        values,
        observed,
        outages,
        pairs,
        torch.from_numpy(adjacency.astype(np.float32)).to(DEVICE),
        torch.tensor([[0.0, 0.0], [1.0, 0.0], [0.0, 1.0]], device=DEVICE),
        torch.zeros(len(FEATURES), device=DEVICE),
        torch.ones(len(FEATURES), device=DEVICE),
    )
    assert directed_output.shape == (len(pairs), len(TARGETS))
    rng = np.random.default_rng(42)
    train_values = rng.normal(size=(300, 3, len(FEATURES))).astype(np.float32)
    train_observed = np.ones_like(train_values, dtype=bool)
    train_pairs = np.column_stack(
        [
            rng.integers(160, 270, size=48),
            rng.integers(0, 3, size=48),
            np.full(48, 6),
        ]
    )
    validation_pairs = train_pairs[:12].copy()
    train_outages = np.zeros((300, 3), dtype=bool)
    ridge = fit_ridge(
        train_values, train_observed, train_outages, train_pairs, DEVICE
    )
    assert ridge.shape[1] == len(TARGETS) and np.isfinite(ridge).all()
    baseline_outputs = evaluate_baselines(
        train_values,
        train_observed,
        train_outages,
        validation_pairs,
        adjacency,
        np.full(len(FEATURES), 3.0, dtype=np.float32),
        np.full(len(FEATURES), 2.0, dtype=np.float32),
        ridge,
    )
    assert set(baseline_outputs) == {
        "persistence",
        "spatial_persistence_k8",
        "knn_mean_k8",
        "linear_both_global_ridge",
    }
    expected_latest = train_values[
        validation_pairs[:, 0, None] - validation_pairs[:, 2, None],
        validation_pairs[:, 1, None],
        TARGET_FEATURE_INDEX[None, :],
    ]
    assert np.allclose(baseline_outputs["persistence"], expected_latest)
    assert np.isfinite(baseline_outputs["spatial_persistence_k8"]).all()
    binary_neighbors = adjacency.copy()
    np.fill_diagonal(binary_neighbors, 0.0)
    for pair_index, (time, target, _) in enumerate(validation_pairs):
        neighbors = binary_neighbors[target] > 0
        expected_spatial = train_values[
            time, neighbors, :
        ][:, TARGET_FEATURE_INDEX].mean(axis=0)
        assert np.allclose(
            baseline_outputs["spatial_persistence_k8"][pair_index],
            expected_spatial,
            atol=1e-5,
        )
    tiny_settings = RunSettings((6,), (("random", 0.3),), (0,), 48, 12, 12, 2)
    trained = train_model(
        GapFillGCN(
            len(FEATURES) * 2 + 1, HISTORY_LENGTH * len(TARGETS) * 2, 8
        ),
        train_pairs,
        validation_pairs,
        train_values,
        train_observed,
        train_outages,
        torch.from_numpy(adjacency.astype(np.float32)).to(DEVICE),
        torch.tensor([[0.0, 0.0], [1.0, 0.0], [0.0, 1.0]], device=DEVICE),
        torch.zeros(len(FEATURES), device=DEVICE),
        torch.ones(len(FEATURES), device=DEVICE),
        tiny_settings,
        19,
    )
    prediction = predict_model(
        trained,
        train_values,
        train_observed,
        train_outages,
        validation_pairs,
        torch.from_numpy(adjacency.astype(np.float32)).to(DEVICE),
        torch.tensor([[0.0, 0.0], [1.0, 0.0], [0.0, 1.0]], device=DEVICE),
        torch.zeros(len(FEATURES), device=DEVICE),
        torch.ones(len(FEATURES), device=DEVICE),
    )
    assert prediction.shape == (len(validation_pairs), len(TARGETS))
    assert np.isfinite(prediction).all()
    scored = score_predictions(
        prediction,
        train_values,
        validation_pairs,
        np.zeros(len(FEATURES), dtype=np.float32),
        np.ones(len(FEATURES), dtype=np.float32),
    )
    assert set(scored) == set(TARGETS)
    assert all(np.isfinite(metric).all() for metric in (np.asarray(pair) for pair in scored.values()))
    print("Gap-fill indexing, target masking, peer outages, graph, and model smoke checks passed.")


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--build-kaggle-notebook", action="store_true")
    parser.add_argument("--smoke-test", action="store_true")
    args = parser.parse_args()
    if args.build_kaggle_notebook:
        print(create_kaggle_notebook())
    elif args.smoke_test:
        run_smoke_tests()
    else:
        run_study()

In [3]:
results = run_study()

Single-turbine gap filling | mode=full | device=cuda

Scenario: gap=1 steps, peer pattern=random, missing=0%, seed=0
Training paper_gru_2022
  epoch 01/10: validation MSE=0.12581
  epoch 02/10: validation MSE=0.10097
  epoch 03/10: validation MSE=0.09020
  epoch 04/10: validation MSE=0.08623
  epoch 05/10: validation MSE=0.08491
  epoch 06/10: validation MSE=0.08391
  epoch 07/10: validation MSE=0.08347
  epoch 08/10: validation MSE=0.08325
  epoch 09/10: validation MSE=0.08314
  epoch 10/10: validation MSE=0.08309
Training mlp
  epoch 01/60: validation MSE=0.08432
  epoch 02/60: validation MSE=0.07748
  epoch 03/60: validation MSE=0.07769
  epoch 04/60: validation MSE=0.06528
  epoch 05/60: validation MSE=0.06121
  epoch 06/60: validation MSE=0.06526
  epoch 07/60: validation MSE=0.06635
  epoch 08/60: validation MSE=0.05517
  epoch 09/60: validation MSE=0.05983
  epoch 10/60: validation MSE=0.05896
  epoch 11/60: validation MSE=0.05613
  epoch 12/60: validation MSE=0.05394
  epoch 13